# Ensaio 2 · A falha de esfera com três métodos

**Estudo:** Smith & Randall (2015). Método 1 = envelope do sinal bruto. Método 2 = pré-branqueamento por cepstro (Borghesani et al., 2013). Método 3 = benchmark; aqui **simplificado**: só a escolha de faixa por curtose espectral, sem a separação discreto/aleatório.

**Hipótese:** com o Método 2 ou o 3, BSF ou 2×BSF entra no top 3 em pelo menos 1 dos 3 tamanhos de falha.

**Dados:** CWRU 12 kHz, lado acionado, 0 hp: 118 (0,007"), 185 (0,014"), 222 (0,021") e 97 (sem defeito).

In [1]:
import sys, json
sys.path.insert(0, "..")
import numpy as np
import matplotlib.pyplot as plt
from src.envelope import *

In [2]:
REC = {97: "sem defeito", 118: 'esfera 0,007"', 185: 'esfera 0,014"', 222: 'esfera 0,021"'}
tabela = {}
for n, label in REC.items():
    x, rpm = load_de(DATA_DIR / f"{n}.mat")
    fr, ff = fault_freqs(rpm)
    sk, lo, hi = kurtosis_band(x)
    metodos = {"M1 envelope": envelope_spectrum(x),
               "M2 cepstro": envelope_spectrum(cepstrum_prewhiten(x)),
               "M3 curtose": envelope_spectrum(x, band=(lo, hi))}
    tabela[n] = {m: {k: peak_rank(f, S, ff[k]) for k in ["BSF", "2xBSF"]} for m, (f, S) in metodos.items()}
    tabela[n]["faixa_curtose_hz"] = (round(lo), round(hi))
for n, r in tabela.items():
    print(n, REC[n], r)
json.dump({str(k): v for k, v in tabela.items()}, open("../results/ensaio2.json", "w"), indent=1)

97 sem defeito {'M1 envelope': {'BSF': (71.18202166124179, 19), '2xBSF': (142.85597159934082, 107)}, 'M2 cepstro': {'BSF': (71.3787929719847, 56), '2xBSF': (139.264895178283, 30)}, 'M3 curtose': {'BSF': (69.46027269224147, 206), '2xBSF': (142.1672720117407, 63)}, 'faixa_curtose_hz': (4575, 5175)}
118 esfera 0,007" {'M1 envelope': {'BSF': (71.86039112024866, 3), '2xBSF': (143.72078224049733, 29)}, 'M2 cepstro': {'BSF': (71.76248868003034, 16), '2xBSF': (142.93756271875077, 41)}, 'M3 curtose': {'BSF': (71.66458623981202, 38), '2xBSF': (143.5249773600607, 17)}, 'faixa_curtose_hz': (2325, 2925)}
185 esfera 0,014" {'M1 envelope': {'BSF': (71.4016053050572, 52), '2xBSF': (141.62139093609966, 29)}, 'M2 cepstro': {'BSF': (71.4016053050572, 92), '2xBSF': (142.11381580027248, 14)}, 'M3 curtose': {'BSF': (71.79554519639545, 118), '2xBSF': (141.62139093609966, 21)}, 'faixa_curtose_hz': (4950, 5550)}
222 esfera 0,021" {'M1 envelope': {'BSF': (69.34937823282046, 17), '2xBSF': (140.2726430638326, 8)}

## Resultado

| Arquivo | M1 envelope | M2 cepstro | M3 curtose |
|---|---|---|---|
| 118 · esfera 0,007" | **BSF 3º** | 16º | 17º |
| 185 · esfera 0,014" | 29º | 14º | 21º |
| 222 · esfera 0,021" | 8º | **2×BSF 1º** | 7º |
| 97 · sem defeito | nenhuma frequência da esfera no top 3 |||

**2 acertos em 9.** A hipótese se confirma, mas a esfera continua sendo a falha mais difícil, e nenhum método vence sempre.

**Limites:** um único motor de bancada, defeitos artificiais, Método 3 simplificado e a tabela do artigo não conferida linha a linha (paywall).